# Advanced usage guide

This ipython notebook provides an advanced usage guide for the library. 

## 1. Agent custom setup

Instead of using the `setup_agent` function, you can create a custom agent setup by directly instantiating the `Agent` class and configuring it with dependencies. 

Below example creates an agent with a dummy display and registers a simple addition function to the agent's toolbox. 

The agent execution always returns a `Result` object, which can be unwrapped to get the final answer.

In [1]:
from xun import Agent, ToolBox, NullDisplay

def add(a: int, b: int) -> int:
    return a + b

agent = Agent(
    display=NullDisplay(), 
    toolbox=ToolBox().register(add)
    )
answer = agent.instruct("What is 2 + 3?").execute()

print(answer.unwrap())



2 + 3 is 5.


## 2. The event-driven display interface
The agent always accepts a `DisplayAbstract` object as a display interface for IO.

The framework provides three display implementations: `Display` (default), `NullDisplay`, and `WebDisplay`.
Where:
- `Display` is a simple console display that prints events to the console.
- `NullDisplay` is a dummy display that does nothing.
- `WebDisplay` is a chat-based web display that can be used as a web application.

To define a custom display, you can implement the `DisplayAbstract` interface with only two methods: 
- `on_event`: called when an event occurs, such as a tool message. 
- `get_choice`: called when the agent needs to make a choice from a list of options.

For example, below we define a custom display that prints the raw event to the console.

In [2]:
from xun import DisplayAbstract

class RawDisplay(DisplayAbstract):
    def on_event(self, event):
        content = event.event
        print(f"Event type: {type(content).__name__}, content: {content}")

    def get_choice(self, *args, **kwargs) -> str:
        raise NotImplementedError("For demonstration purposes, this method is not implemented.")

# the with statement triggers an `AgentUnbind` event to the display
# which can also be called via `agent.finalize()`
with Agent(display=RawDisplay()) as agent:
    agent.toolbox.register(add)
    _ = agent.instruct("What is 2 + 3?").execute()

Event type: AgentBindEvent, content: 
Event type: UserMessageEvent, content: content='What is 2 + 3?' images=[]
Event type: ModelWorkingEvent, content: model_call_id='299dcb78-63fc-490d-bdf1-b056c6745453' remaining_iterations=64
Event type: ToolCallEvent, content: tool_call_id='chatcmpl-tool-8237f6a5450ff0d0' tool_name='add' args={'a': 2, 'b': 3}
Event type: ToolResultEvent, content: tool_call_id='chatcmpl-tool-8237f6a5450ff0d0' result=5
Event type: ModelWorkingEvent, content: model_call_id='281c70a4-cc13-480c-81c7-151d2cf560dc' remaining_iterations=63
Event type: ModelMessageEvent, content: model_call_id='281c70a4-cc13-480c-81c7-151d2cf560dc' content='\n\n2 + 3 is 5.'
Event type: AgentUnbindEvent, content: 


## 3. Tool attributes

We can attach metadata to a tool function, for example, to change the tool name.

In [3]:
from xun import tool_attr, setup_agent

@tool_attr(name="MultiplyTool")
def multiply(a: int, b: int) -> int:
    return a * b

agent = setup_agent(tools=[multiply], display=NullDisplay())
agent.instruct("What is 2 * 3?").execute()

tool_name = agent.instruct("What tool did you use?").execute().unwrap()

print(f"Reply from agent: {tool_name}")

Reply from agent: 

I used the **MultiplyTool** to calculate the result.


## 4. Structured output

The framework support pydantic model validation for tool output. 
You can define a pydantic model and use it as the parameter for the execution. 

- There will be prompt injected to the message, informing the agent of the correct format. 
- The output will be automatically validated and deduced for type hinting. 

In [4]:
from pydantic import BaseModel

class StoryModel(BaseModel):
    title: str
    content: str

story = setup_agent(display = NullDisplay()).instruct(
    "Write a short story within 100 words."
).execute(schema=StoryModel).unwrap()

print(f"Title: {story.title}")
print(f"Content: {story.content}")

Title: The Last Light
Content: The lighthouse keeper lit the final wick. Waves crashed against the crumbling stone base, but the beam cut through the fog, steady and bright. For fifty years, he had guided ships home. Tonight, for the first time, the horizon remained empty. He smiled, blowing out the flame. His watch was over. The darkness didn't feel like an end, but a long-awaited rest. He closed his eyes, listening to the rhythm of the sea one last time, and let the night take him.


## 5. Tool execution context

While a tool is just a plain Python function without state, 
the framework provides an execution context for it.

Simply declare a `ToolCallContext` parameter in the tool function, and the framework will automatically inject the context object when the tool runs.

The context includes several built-in attributes, and you can also attach your own values when invoking the tool.

In [5]:
from xun import ToolCallContext as Context
import datetime

def tool_with_context(context: Context[dict]):
    print("Tool called by: ", context.agent.name)
    context.value["tool_call_time"] = datetime.datetime.now().strftime("%H:%M:%S")

context_value = {"tool_call_time": "?"}
setup_agent(
    name = "ContextAgent",
    tools=[tool_with_context], 
    display=NullDisplay()
).instruct(
    "Call the only tool you have in hand."
).execute(context=context_value)

print(f"Context after tool execution: {context_value}")

Tool called by:  ContextAgent
Context after tool execution: {'tool_call_time': '19:45:27'}


## 6. Sub-agent spawning

It is easy to make agent spawnning a tool, just to create a new agent in the function to execute the tool call. 

As another approach, the framework provides a quick and generic way to quickly setup sub-agent spawning. 
By using the `ToolBox::with_subagent_provider` function, you register a generic sub-agent call tools to execute tasks.

This function support declare an agent-getter function, if its omitted, the framework will use the default agent setup to spawn a sub-agent (inheriting the parent agent's toolbox and display).

In [6]:
import math

def sqrt(ctx: Context, a: float) -> float:
    print(f"Tool called by: {ctx.agent.name}")
    return math.sqrt(a)

def agent_getter(_):
    return setup_agent(display=NullDisplay(), tools=[sqrt])

agent = Agent(
    display = NullDisplay(), 
    toolbox = ToolBox().with_subagent_provider(agent_getter)
    )
result = agent.instruct("What is the result of square root of 114514? call a sub-agent to calculate it.").execute()

print(f"Result from sub-agent: {result.unwrap()}")

Tool called by: subagent
Result from sub-agent: 

The square root of 114514 is approximately **338.40**.


## 7. Lifecycle hooks

We can register hooks to the agent execution.

In [7]:
from xun import Hooks
hooks = Hooks()
hooks.before_tool_call.add(lambda arg: print(arg.tool_calls))
hooks.after_tool_call.add(lambda arg: print(arg.tool_results))
hooks.after_initialize.add(lambda arg: print(arg.agent.name, "has been initialized."))
hooks.before_finalize.add(lambda arg: print(arg.agent.name, "is about to finalize."))

agent = Agent(
    name = "HookAgent",
    display = NullDisplay(),
    toolbox = ToolBox().register(add),
    hooks = hooks
    )

agent.instruct("What is 2 + 3?").execute()
agent.finalize()

HookAgent has been initialized.
[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-839205d3de3ce42d', function=Function(arguments='{"a": 2, "b": 3}', name='add'), type='function')]
[('chatcmpl-tool-839205d3de3ce42d', <xun.types.Result object at 0x773467707710>)]
HookAgent is about to finalize.


## 8. Cancellation

From time to time, we may want to cancel long-running tasks. 

To do this, two things are needed: 
- implement check points in the code (for long running tool), and
- trigger the cancel event when needed.

The cancel also supports sharing between parent and child agents. When a parent agent is cancelled, all its child agents will also be cancelled.

In [ ]:
import time
import threading
from xun import CancelledError

def long_running_task(ctx: Context):

    # emulate user canceling the sub-agent after 2 seconds
    global parent_agent
    def cancel_from_parent():
        time.sleep(2)
        parent_agent.cancel()
    threading.Thread(target=cancel_from_parent, daemon=True).start()

    for i in range(10):
        print(f"Sub-agent is working... step {i+1}/10")
        time.sleep(1)
        ctx.agent.check_cancel()

def subagent_getter(ctx: Context) -> Agent:
    # must use inherit to get subagent, 
    # or pass parent.cancel_event to subagent
    agent = Agent.inherit(ctx.agent).system(
        "Whatever task you get, just call your only tool."
    )
    agent.toolbox.register(long_running_task)
    return agent

parent_agent = Agent(
    display = NullDisplay(),
    toolbox = ToolBox().with_subagent_provider(
        subagent_getter,
    ))

try:
    parent_agent.instruct("Let subagent call its tool (just input 'start')").execute()
except CancelledError:
    print("Tada! The sub-agent was cancelled by the parent agent. All tasks are stopped.")

Sub-agent is working... step 1/10
Sub-agent is working... step 2/10
Tada! The sub-agent was cancelled by the parent agent. All tasks are stopped.


## 9. Web display service

We have a built-in web interface for the agent. 

It supports multiplexing different agents in a single web service, and each agent can have its own display and toolbox.

In [9]:
from xun import WebDisplay, WebDisplayService
from xun import setup_agent

display1 = WebDisplay(expose_files=True)
setup_agent(display=display1, default_tools=True, workdir='.test')
setup_agent(display=display1, default_tools=True, workdir='.tmp')

display2 = WebDisplay(expose_files=False)
setup_agent(display=display2, default_tools=True, workdir='.test')

service = WebDisplayService(port=8877).mount("/display1", display1).mount("/display2", display2)
service.start(blocking=False)

# you can now access the web interface at following URLs

Agents are available at the following URLs:
http://localhost:8877/display1/?token=WeUMmPZrvLDodLYK9gn19INM8xWhAkf6
http://localhost:8877/display2/?token=WeUMmPZrvLDodLYK9gn19INM8xWhAkf6


<Thread(xun-web-server, started daemon 131066956346944)>

In [10]:
service.stop()